# Homework: File conversion & Trace header fixing

This notebook prompts you to read a single **SEISAN** waveform file:

`2001-02-02-0303-55S.MVO___019`

and convert it into processed MiniSEED and SAC outputs, using ObsPy. Along the way you will practice:

* parsing information from filenames
* reading SEISAN waveforms
* editing Trace metadata
* tapering & filtering
* unit scaling using a sensitivity (or calibration) value
* changing the sampling rate
* writing MiniSEED and SAC

You may NOT use AI to help with this.

**Important: Make sure you read all the way to the bottom of these instructions before you start this exercise!**

## 0) Setup

Import read and UTCDateTime from obspy

Set a string variable called 'filename' that contains the path to the Seisan file, e.g.

```
filename = "/Users/thompsong/Developer/CompSciS26/week5/2001-02-02-0303-55S.MVO___019"
```

In [ ]:
import obspy 
from obspy import UTCDateTime
from obspy import read

filename = "/Users/izzy/Downloads/Computational Seismology/CompSciS26/week5/2001-02-02-0303-55S.MVO___019"
st = read(filename)
print(st)

Good, though I didn't ask you to read it yet. 3/3


## 1) Parse the filename
Given the input filename like YYYY-MM-DD-hhmmssS.MVO__NUM, e.g. 2003-06-05-1845-15S.PVO___022 extract:

1.	Start time from the prefix formatted as YYYY-MM-DD-hhmm-ss and create a UTCDateTime from it.
<br/>Example: parse 2003-06-05-1845-15 into a UTCDateTime.

2.	The format code: the single letter after the time is S meaning SEISAN.

3.	The number of channels: the last 3 digits (e.g. 022) → integer number of channels.

Remember you can index and subset strings. 

Print a one-line summary like:

```Parsed starttime=2003-06-05T18:45:15Z, format=S, nchans=22```


In [ ]:
from pathlib import Path

path = Path(filename)
file_basename = path.name

time_part = file_basename[0:18]
start_time = UTCDateTime(time_part)
format_code = file_basename[18]
nchans = int(file_basename[-3:])

print(f"Parsed starttime={start_time}, format={format_code}, nchans={nchans}")




Good 8/8

## 2) Read the SEISAN file

Read the file into an ObsPy Stream.
* Verify that the number of traces in the stream matches the parsed channel count.
* If it does not match, print a warning and continue.

In [ ]:
st = read(filename)

count = len(st)
if count != nchans:
    print(f"Warning: expected {nchans} channels but found {count} in file {filename}")
else:
    print(f"Found expected number of channels ({nchans}) in file {filename}")


Good 6/6

## 3) Fix the SEED codes

1) These all have a blank network code. Set network code to 'MV'.

2) The location code is bogus. Change it from 'J' to '00'.

3) The channel code is not SEED compliant. Make the following changes:
    * "PRS" -> "SDO" (barometer)
    * "SB" -> "BH"
    * "S " -> "SH"

In [ ]:
for tr in st: 
    tr.stats.station = tr.stats.station.strip()
    
    tr.stats.network = 'MV'

    if tr.stats.location == 'J':
        tr.stats.location = '00'

    if "PRS" in tr.stats.channel:
        tr.stats.channel = tr.stats.channel.replace("PRS", "SDO")
    elif "SB" in tr.stats.channel:
        tr.stats.channel = "BH" + tr.stats.channel[-1]
    elif "S" in tr.stats.channel:
        tr.stats.channel = "SH" + tr.stats.channel[-1]

print(st)

Almost. This works, but you got lucky with the last step. You should be checking for "S " not just "S". 7/8

## 4) Preprocess each trace

Apply these steps in this order:

1.	Taper: 5% taper

2.	High-pass filter: 0.5 Hz, two-way (zero-phase)

3.	Scale to physical units using an overall sensitivity of 8,000,000 Counts per (m/s). The trace data are currently in Counts. Convert them to m/s.


In [ ]:

st.taper(max_percentage=0.05)

st.filter("highpass", freq=0.5, zerophase=True)

for tr in st: 
    tr.data = tr.data/8000000


print(f"Max values for first three traces: {[tr.data.max() for tr in st[:3]]}")


10/10

## 5) Round the sampling rate
Change each trace sampling rate to the nearest integer (in samples/sec). 

In [ ]:
for tr in st: 
    new_sampling_rate = round(tr.stats.sampling_rate)
    tr.stats.sampling_rate = new_sampling_rate

print(f"New sampling rate: {st[0].stats.sampling_rate}")

3/3

## 6) Force the start time

Set the start time for every trace to:<br/>
```UTCDateTime(‘2001-02-02T03:00:00’)```


In [ ]:
forced_time = UTCDateTime("2001-02-02T03:00:00")
for tr in st: 
    tr.stats.starttime = forced_time

print(st[5].stats.starttime)
print(st[11].stats.starttime)

3/3

## 7A) Write MiniSEED

1. Output filename should match the input naming style, but replace the S with M. But remember that the start time has changed.

2. Write stream as MiniSEED.

3. Re-read the MiniSEED file and plot it.



In [ ]:
output_file = "2001-02-02-0300-00M.MVO___019.mseed"
st.write(output_file, format="MSEED")
st_final = read(output_file)
st_final.plot();

No need to "hardcode" the output filename. Can just do:

```
output_file = forced_time.strftime('%Y-%M-%D-%H%M-%S-%f') + 'M' + filename[-9:]
```

Also did not ask for '.mseed' on the end. MiniSEED is implied by the 'M.' in Seisan naming convention.

3/5

## 7B) Write SAC

1. Write SAC files (one per trace).

2. Put them in a folder like SAC/.

3. Filenames should include NET.STA.LOC.CHA and optionally start time.

4. Re-read the SAC files and plot them.


In [ ]:
import os

folder_name = "SAC" 

for tr in st: 
    file_id = f'{tr.stats.network}.{tr.stats.station}.{tr.stats.location}.{tr.stats.channel}'
    time_str = tr.stats.starttime.strftime("%Y-%m-%dT%H:%M:%S")
    filename= f"{folder_name}/{file_id}_{time_str}.SAC"
    tr.write(filename, format="SAC")
print(f"Saved {len(st)} SAC files in folder '{folder_name}'.")

st_sac = read(f"{folder_name}/*.SAC")
print(st_sac)

st_sac.plot();


3/4. 
Easier would have been:
```file_id = tr.id```

# Deliverables

Submit:

1.	Your Python script or notebook (hw_seisan_convert.py or .ipynb)

2.	Screenshots of the stream plots of the MiniSEED and SAC files you read back in.


---

## Hints:

Ignore what I wrote above about AI. You CAN use AI for this exercise. But make sure you understand the code it writes! You might be asked to explain it in class!

Remember that strings can be indexed in just the same way as lists. If you want the last 4 characters, you can do:

```
my_str = “my_filename.123”
last_4 = my_str[-4:]
```

Or you can split on “_” using:
```
my_list = my_str.split(“_”)
```

You can round a float(ing point variable) to the nearest integer with the round() function:

```
a = 75.19
b = round(a)
print(b) # b=75
```

You can build a UTCDateTime object from a string by specifying the string format, e.g.:

```
import obspy
filename = “2022-05-08-1715-05M.PVO__006”
filetime = obspy.UTCDateTime.strptime(filename, “%Y-%m-%d-%H%M-%S”)
```

Read a waveform file like this:

```
import obspy
st = obspy.read(filename) # read outputs an ObsPy Stream object
```

Remember you can loop over Trace objects in a Stream object like this:

```
for tr in st:
    # THIS IS A COMMENT do something to tr
```

You can taper and filter traces with:

```
tr.taper()
tr.filter()
```

A Trace object has a stats attribute, and a data attribute.

```
tr.data # this is a numpy array – the actual sequence of sample values in the timeseries
tr.stats # this is a dictionary with attributes such as:
    #	sampling_rate
    #	network
    #	station
    #	location
    # 	channel
```

To scale a numpy array called ‘a’ by a factor of 5 you can do:

$$f = 5$$
$$a = a * f$$

Remember that tr.data is a numpy array.
But also see what happens if you just try to multiple tr directly:

```tr = tr * 5```

You can write a Stream object to a file using:
st.write(‘/path/to/filename.ext’, format=’<format>’)
Where <format> is ‘mseed’, or ‘sac’, or ‘seisan’, or some other format ObsPy supports.
You can also write a Trace object to a file in exactly the same way.




